In [1]:
import boto3
from botocore.config import Config
from io import StringIO
import pandas as pd
import torch

In [ ]:
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()
read_access_key = os.getenv("AWS_ACCESS_KEY_ID")
read_secret_key = os.getenv("AWS_SECRET_ACCESS_KEY")
bucket_name = os.getenv("AWS_BUCKET_NAME")
endpoint_url = os.getenv("AWS_ENDPOINT_URL")
file_key = "dar_tables/full_dataset_sequences.txt" 
local_file_path = 'dataset'

# Initialize S3 client
s3 = boto3.client(
    "s3",
    aws_access_key_id=read_access_key,
    aws_secret_access_key=read_secret_key,
    endpoint_url=endpoint_url,
    config=Config(signature_version="s3v4"),
)


def fetch_dataset_from_s3(bucket_name, file_key):
    """
    Fetch dataset from a text file in an Amazon S3 bucket.
    :param bucket_name: Name of the S3 bucket.
    :param file_key: Path to the file in the bucket.
    :return: List of sequences and features.
    """
    print(f"Fetching file from S3: {file_key}")
    obj = s3.get_object(Bucket=bucket_name, Key=file_key)
    file_content = obj["Body"].read().decode("utf-8")
    
    # Assuming the file is tab-delimited with a column named 'sequence'
    df = pd.read_csv(StringIO(file_content), sep="\t")

    print(f"Loaded {len(df)} sequences from the S3 file.")
    return df

def save_df_to_s3(df, bucket_name, s3_folder, file_name):
    """
    Save a DataFrame to a specified folder in the S3 bucket as a CSV file.
    :param df: DataFrame to save.
    :param bucket_name: Name of the S3 bucket.
    :param s3_folder: Folder path within the bucket.
    :param file_name: Name of the file to save in the bucket.
    """
    s3_key = f"{s3_folder}/{file_name}"
    try:
        # Convert DataFrame to CSV in memory
        csv_buffer = StringIO()
        df.to_csv(csv_buffer, index=False)
        
        # Upload CSV to S3
        s3.put_object(Bucket=bucket_name, Key=s3_key, Body=csv_buffer.getvalue())
        print(f"File successfully uploaded to S3: {bucket_name}/{s3_key}")
    except Exception as e:
        print(f"Error uploading to S3: {e}")


In [3]:
df = fetch_dataset_from_s3(bucket_name, file_key)

Fetching file from S3: dar_tables/full_dataset_sequences.txt
Loaded 801880 sequences from the S3 file.


In [4]:
# Create the new column 'cisbin_present'
df['cisbin_present'] = df['CisbinChr'].notna().astype(int)

In [5]:
# columns to select
col_select = ['cell_type',
              'Fed',
              'Fasted',
              'Refed',
                'length',
                'gene_present',
                'peak',
                'gene_number',
                'promoter_number',
                'cisbin_present']


df = df[col_select]

In [6]:
df.head()


,cell_type,Fed,Fasted,Refed,length,gene_present,peak,gene_number,promoter_number,cisbin_present
0,Endothelial,1,0,0,164,1,peak_0,1,1,1
1,Endothelial,0,1,0,234,1,peak_1,1,1,1
2,Endothelial,0,1,0,316,1,peak_2,1,1,1
3,Endothelial,1,0,1,595,1,peak_3,1,1,1
4,Endothelial,0,0,1,244,0,peak_4,1,1,0


In [10]:
# Save DataFrame to S3
save_df_to_s3(df, bucket_name, s3_folder="dataset", file_name="features.csv")

File successfully uploaded to S3: metabolic-atac-peaks/dataset/features.csv


## Done